# multi_1h_jma_atr_012

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (173).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 14 |
| Detected functions | resample_1hr_custom, create_example_configs, atr, compute_jma_indicator, __init__, calculate_position_size, get_stop_loss_distance, __init__, create_trade_record, __init__, run_backtest, _is_invalid_row |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Binance_Crypto_T5_2025.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2023-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['XRPUSDT', 'SOLUSDT', 'ETHUSDT', 'BTCUSDT', 'BNBUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
# Create example configurations with enhanced parameters
def create_example_configs():
    """Create example ticker configurations for enhanced backtest"""

    ticker_configs = [
        TickerConfig(
            ticker='SOLUSDT',
            jma_params=JMAParameters(
                smooth_length=14,
                normalization_period=36,
                upper_threshold=1.75,
                lower_threshold=-1.25,
                rs_period=14,
                rs_percentile=10
            ),
            slippage_bps=7,
            fee_rate=0.00025
        ),

        TickerConfig(
            ticker='BTCUSDT',
            jma_params=JMAParameters(
                smooth_length=14,
                normalization_period=60,
                upper_threshold=1.5,
                lower_threshold=-1,
                rs_period=3,
                rs_percentile=10
            ),
            slippage_bps=2,
            fee_rate=0.00015
        ),

        TickerConfig(
            ticker='XRPUSDT',
            jma_params=JMAParameters(
                smooth_length=14,
                normalization_period=36,
                upper_threshold=1.5,
                lower_threshold=-1.0,
                rs_period=14,
                rs_percentile=10
            ),
            slippage_bps=6,
            fee_rate=0.00025
        ),

        TickerConfig(
            ticker='ETHUSDT',
            jma_params=JMAParameters(
                smooth_length=14,
                normalization_period=60,
                upper_threshold=1.75,
                lower_threshold=-1.25,
                rs_period=14,
                rs_percentile=10
            ),
            slippage_bps=3,
            fee_rate=0.0002
        ),

        TickerConfig(
            ticker='BNBUSDT',
            jma_params=JMAParameters(
                smooth_length=14,
                normalization_period=36,
                upper_threshold=1.75,
                lower_threshold=-1.25,
                rs_period=3,
                rs_percentile=10
            ),
            slippage_bps=4,
            fee_rate=0.00025
        )
    ]

    return ticker_configs

In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Optional

# =====================================
# Data Classes
# =====================================

@dataclass
class PortfolioConfig:
    """Portfolio-level configuration parameters"""
    initial_capital: float = 500_000
    max_risk_per_trade: float = 2000        # Fixed $2000 risk per trade
    max_daily_drawdown: float = 5000        # $5000 daily drawdown limit
    max_portfolio_exposure: float = 1.0     # 100% max allocation
    fee_rate: float = 0.001                 # 0.1% trading fee
    slippage_bps: float = 2.0               # 2 basis points


@dataclass
class JMAParameters:
    """Parameters for JMA indicator"""
    smooth_length: int = 14
    normalization_period: int = 36
    upper_threshold: float = 1.5
    lower_threshold: float = -1.0
    rs_period: int = 14
    rs_percentile: float = 10.0


# =====================================
# Technical Indicators
# =====================================

class TechnicalIndicators:
    """Math-based indicator calculations (no TA libraries)."""

    @staticmethod
    def atr(data: pd.DataFrame, window: int = 14) -> pd.Series:
        """Average True Range (ATR) using pure math."""
        high, low, close = data["high"], data["low"], data["close"]

        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()

        true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = true_range.rolling(window=window, min_periods=1).mean()

        return atr


def compute_jma_indicator(df: pd.DataFrame, params: JMAParameters) -> pd.DataFrame:
    """
    Compute Jurik Moving Average (JMA) with volatility filter
    and generate trading signals.
    """
    if df is None or df.empty:
        return df

    df = df.copy()
    required_cols = ["open", "high", "low", "close"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        print(f"[Warning] Missing columns: {missing_cols}")
        return df

    try:
        # === Jurik Moving Average ===
        price = df["close"]
        alpha = 2.0 / (params.smooth_length + 1)

        jma = [price.iloc[0]]
        for p in price.iloc[1:]:
            jma.append(jma[-1] + (p - jma[-1]) * alpha)
        jma_series = pd.Series(jma, index=price.index)

        # === Normalization ===
        rolling_mean = jma_series.rolling(params.normalization_period).mean()
        rolling_std = jma_series.rolling(params.normalization_period).std()
        normalized = (jma_series - rolling_mean) / rolling_std

        # === Rogers-Satchell Volatility ===
        o, h, l, c = df["open"].replace(0, np.nan), df["high"], df["low"], df["close"].replace(0, np.nan)
        log_hc = np.log(h / c)
        log_ho = np.log(h / o)
        log_lc = np.log(l / c)
        log_lo = np.log(l / o)

        rs_vol = (log_hc * log_ho + log_lc * log_lo).rolling(params.rs_period).mean()
        rs_threshold = rs_vol.rolling(params.normalization_period).quantile(params.rs_percentile / 100.0)

        # === Signals ===
        longs = (normalized > params.upper_threshold) & (rs_vol > rs_threshold)
        shorts = (normalized < params.lower_threshold) & (rs_vol > rs_threshold)

        signal = pd.Series(0, index=df.index)
        signal[longs] = 1
        signal[shorts] = 2

        # Attach results
        df["JMA"] = jma_series
        df["Normalized_JMA"] = normalized
        df["RS_Volatility"] = rs_vol
        df["RS_Threshold"] = rs_threshold
        df["Signal"] = signal

        return df

    except Exception as e:
        print(f"[Error] JMA calculation failed: {e}")
        df["JMA"] = df["close"]
        df["Normalized_JMA"] = 0
        df["RS_Volatility"] = 0
        df["RS_Threshold"] = 0
        df["Signal"] = 0
        return df


# =====================================
# Position Sizer
# =====================================

class PositionSizer:
    """
    Dynamic position sizing with fixed $2000 risk per trade:
    Quantity = Fixed_Risk / (ATR_max * Price)
    where ATR_max = max(highest(ATR14, 20)/2 , highest(ATR5, 20)/2)
    """

    def __init__(self, config: PortfolioConfig):
        self.config = config

    def calculate_position_size(self, df: pd.DataFrame, price: float, equity: float) -> int:
        """
        Calculate position size using fixed $2000 risk per trade.
        The position size is dynamically adjusted based on ATR to maintain constant dollar risk.
        """
        try:
            if price <= 0 or equity <= 0 or df.empty:
                return 0

            # Compute ATR(14) & ATR(5)
            atr14 = TechnicalIndicators.atr(df, 14)
            atr5 = TechnicalIndicators.atr(df, 5)

            # Highest ATR values over last 20 bars (shifted by 1 to exclude current bar)
            atr14_highest = atr14.shift(1).rolling(20).max()
            atr5_highest = atr5.shift(1).rolling(20).max()

            # ATR_max = maximum of the two ATR measures divided by 2
            atr_max = np.maximum(atr14_highest / 2.0, atr5_highest / 2.0)
            current_atr_max = atr_max.iloc[-1]

            if pd.isna(current_atr_max) or current_atr_max <= 0:
                return 0

            # Fixed $2000 risk per trade
            fixed_risk = self.config.max_risk_per_trade  # $2000

            # Position size = Risk / (Stop Loss Distance * Price)
            # Stop Loss Distance = ATR_max (our volatility-based stop)
            raw_qty = fixed_risk / (current_atr_max * price)

            # Ensure we don't exceed portfolio exposure limits
            max_position_value = equity * self.config.max_portfolio_exposure
            max_qty_by_exposure = max_position_value / price

            # Take the minimum of risk-based and exposure-based limits
            final_qty = min(raw_qty, max_qty_by_exposure)

            return max(0, int(final_qty))

        except Exception as e:
            print(f"[Error] Position sizing failed: {e}")
            return 0

    def get_stop_loss_distance(self, df: pd.DataFrame) -> float:
        """Get the current ATR-based stop loss distance for reference."""
        try:
            atr14 = TechnicalIndicators.atr(df, 14)
            atr5 = TechnicalIndicators.atr(df, 5)

            atr14_highest = atr14.shift(1).rolling(20).max()
            atr5_highest = atr5.shift(1).rolling(20).max()

            atr_max = np.maximum(atr14_highest / 2.0, atr5_highest / 2.0)
            return atr_max.iloc[-1]
        except:
            return 0.0


# =====================================
# Example Usage
# =====================================

if __name__ == "__main__":
    # Sample data creation (replace with your actual data)
    np.random.seed(42)
    dates = pd.date_range('2023-01-01', periods=100, freq='D')

    # Create sample OHLC data
    base_price = 100
    returns = np.random.normal(0, 0.02, 100)
    prices = [base_price]
    for ret in returns[1:]:
        prices.append(prices[-1] * (1 + ret))

    data = pd.DataFrame({
        'open': [p * (1 + np.random.normal(0, 0.005)) for p in prices],
        'high': [p * (1 + abs(np.random.normal(0, 0.01))) for p in prices],
        'low': [p * (1 - abs(np.random.normal(0, 0.01))) for p in prices],
        'close': prices
    }, index=dates)

    # Configuration
    config = PortfolioConfig()
    jma_params = JMAParameters()

    print(f"Portfolio Configuration:")
    print(f"Initial Capital: ${config.initial_capital:,.2f}")
    print(f"Risk per Trade: ${config.max_risk_per_trade:,.2f}")
    print(f"Risk as % of Capital: {(config.max_risk_per_trade/config.initial_capital)*100:.2f}%")
    print("="*50)

    # Compute JMA + Signals
    df = compute_jma_indicator(data, jma_params)

    # Position sizing
    sizer = PositionSizer(config)
    current_price = df["close"].iloc[-1]
    current_equity = config.initial_capital

    qty = sizer.calculate_position_size(df, price=current_price, equity=current_equity)
    stop_distance = sizer.get_stop_loss_distance(df)

    # Add position column
    df["Position"] = 0
    if len(df) > 0:
        df.at[df.index[-1], "Position"] = qty

    # Calculate trade metrics
    position_value = qty * current_price
    risk_per_share = stop_distance
    total_risk = qty * risk_per_share

    # Display results
    print(f"\nTrade Analysis:")
    print(f"Current Price: ${current_price:.2f}")
    print(f"Position Size: {qty:,} shares")
    print(f"Position Value: ${position_value:,.2f}")
    print(f"Position as % of Capital: {(position_value/current_equity)*100:.2f}%")
    print(f"ATR Stop Distance: ${stop_distance:.2f}")
    print(f"Actual Risk: ${total_risk:.2f}")
    print(f"Target Risk: ${config.max_risk_per_trade:.2f}")
    print(f"Risk Accuracy: {(total_risk/config.max_risk_per_trade)*100:.1f}%")

    # Show last few rows with signals
    print(f"\nLast 5 rows with signals:")
    display_cols = ["close", "JMA", "Normalized_JMA", "Signal", "Position"]
    available_cols = [col for col in display_cols if col in df.columns]
    print(df[available_cols].tail())

In [ ]:
df

In [ ]:
df.to_csv("/content/drive/MyDrive/Test_Tradebook/tradebook_2022_25.csv")

In [ ]:
# ==========================================
# TRADE EXECUTOR CLASS
# ==========================================

class TradeExecutor:
    """Handles trade execution logic"""

    def __init__(self, config: PortfolioConfig):
        self.config = config
        self.logger = logging.getLogger(f"{__name__}.TradeExecutor")

    def create_trade_record(self, position: dict, exit_price: float, exit_time: pd.Timestamp,
                          exit_reason: str, ticker_config: TickerConfig) -> Optional[TradeRecord]:
        """Create trade record when closing position"""
        try:
            if not position or pd.isna(exit_price) or exit_price <= 0:
                return None

            direction = position.get('direction')
            quantity = position.get('quantity', 0)
            entry_price = position.get('entry_price', 0)

            if not direction or quantity <= 0 or entry_price <= 0:
                return None

            # Calculate gross P&L
            if direction == 'long':
                gross_pnl = (exit_price - entry_price) * quantity
            else:
                gross_pnl = (entry_price - exit_price) * quantity

            # Calculate costs
            notional = (entry_price * quantity) + (exit_price * quantity)
            fee_rate = ticker_config.fee_rate if ticker_config.fee_rate else self.config.fee_rate
            fees = notional * fee_rate
            slippage = notional * (ticker_config.slippage_bps / 10000)

            net_pnl = gross_pnl - fees - slippage

            return TradeRecord(
                ticker=position.get('ticker', ''),
                entry_time=position.get('entry_time'),
                exit_time=exit_time,
                direction=direction,
                entry_price=entry_price,
                exit_price=exit_price,
                quantity=quantity,
                gross_pnl=gross_pnl,
                fees=fees,
                slippage=slippage,
                net_pnl=net_pnl,
                stop_loss=position.get('stop_loss', 0),
                take_profit=position.get('take_profit', 0),
                exit_reason=exit_reason
            )

        except Exception as e:
            self.logger.error(f"Error creating trade record: {e}")
            return None


In [ ]:
# ==========================================
# SINGLE TICKER STRATEGY CLASS
# ==========================================

class SingleTickerStrategy:
    """Handles trading logic for a single ticker"""

    def __init__(self, config: PortfolioConfig):
        self.config = config
        self.position_manager = PositionManager(config)
        self.trade_executor = TradeExecutor(config)
        self.logger = logging.getLogger(f"{__name__}.SingleTickerStrategy")

    def run_backtest(self, ticker: str, data: pd.DataFrame, ticker_config: TickerConfig) -> List[TradeRecord]:
        """Run backtest for single ticker"""
        try:
            if data is None or data.empty:
                self.logger.warning(f"No data for {ticker}")
                return []

            # Initialize trading state
            trades = []
            position = None
            daily_pnl = 0.0
            current_date = None
            daily_limit_reached = False
            exit_next_open = False
            current_equity = self.config.initial_capital
            cumulative_pnl = 0.0
            last_exit_time = None

            data = data.sort_index()
            max_daily_loss = current_equity * self.config.daily_drawdown_limit

            for idx, row in data.iterrows():
                try:
                    # Skip rows with insufficient data
                    if self._is_invalid_row(row):
                        continue

                    # Check for new trading day
                    current_row_date = self._get_trading_date(idx)
                    if current_date != current_row_date:
                        current_date = current_row_date
                        daily_pnl = 0.0
                        daily_limit_reached = False
                        current_equity = self.config.initial_capital + cumulative_pnl
                        max_daily_loss = current_equity * self.config.daily_drawdown_limit

                    # Handle exit on open (delayed SL)
                    if exit_next_open and position:
                        trade = self._handle_exit_on_open(position, row, idx, ticker_config)
                        if trade:
                            trades.append(trade)
                            daily_pnl += trade.net_pnl
                            cumulative_pnl += trade.net_pnl
                            current_equity += trade.net_pnl

                        position = None
                        exit_next_open = False
                        last_exit_time = idx

                        if daily_pnl < -max_daily_loss:
                            daily_limit_reached = True
                        continue

                    # Skip if daily limit reached
                    if daily_limit_reached:
                        continue

                    # Entry logic
                    if self._should_enter_position(position, idx, last_exit_time, daily_limit_reached):
                        entry_result = self._handle_entry(row, idx, ticker, current_equity, ticker_config)

                        if entry_result:
                            new_position, immediate_exit = entry_result

                            if immediate_exit:  # Entry SL hit
                                trade = immediate_exit
                                trades.append(trade)
                                daily_pnl += trade.net_pnl
                                cumulative_pnl += trade.net_pnl
                                current_equity += trade.net_pnl
                                last_exit_time = idx

                                if daily_pnl < -max_daily_loss:
                                    daily_limit_reached = True
                            else:
                                position = new_position
                                # Check for delayed SL exit
                                if self._check_delayed_sl_exit(position, row):
                                    exit_next_open = True

                    # Position management
                    elif position:
                        exit_result = self._manage_position(position, row, idx, ticker_config)

                        if exit_result:
                            trade = exit_result
                            trades.append(trade)
                            daily_pnl += trade.net_pnl
                            cumulative_pnl += trade.net_pnl
                            current_equity += trade.net_pnl

                            position = None
                            last_exit_time = idx

                            if daily_pnl < -max_daily_loss:
                                daily_limit_reached = True

                except Exception as e:
                    self.logger.error(f"Error processing row for {ticker} at {idx}: {e}")
                    continue

            self.logger.info(f"Completed {ticker}: {len(trades)} trades")
            return trades

        except Exception as e:
            self.logger.error(f"Error processing {ticker}: {e}")
            return []

    def _is_invalid_row(self, row) -> bool:
        """Check if row has invalid data"""
        required_fields = ['atr_14', 'Signal', 'open', 'close']
        return any(pd.isna(row.get(field, np.nan)) for field in required_fields)

    def _get_trading_date(self, idx):
        """Extract trading date from index"""
        try:
            return idx.date() if hasattr(idx, 'date') else pd.to_datetime(idx).date()
        except:
            return None

    def _should_enter_position(self, position, idx, last_exit_time, daily_limit_reached) -> bool:
        """Check if should enter new position"""
        return (position is None and
                idx != last_exit_time and
                not daily_limit_reached)

    def _handle_entry(self, row, idx, ticker, current_equity, ticker_config):
        """Handle position entry logic"""
        signal = row.get('Signal', 0)
        direction = None

        if signal == 1:
            direction = 'long'
        elif signal == 2:
            direction = 'short'

        if not direction:
            return None

        # Calculate entry price with slippage
        direction_mult = 1 if direction == 'long' else -1
        entry_price = row['open'] * (1 + self.config.slippage_bps/10000 * direction_mult)

        # Calculate position size
        atr_14 = row.get('atr_14', 0)
        atr_5 = row.get('atr_5', atr_14)

        quantity, stop_loss, take_profit, entry_sl = self.position_manager.calculate_position_size(
            entry_price, atr_14, atr_5, direction, current_equity
        )

        if quantity <= 0:
            return None

        position = {
            'ticker': ticker,
            'entry_time': idx,
            'entry_price': entry_price,
            'direction': direction,
            'quantity': quantity,
            'stop_loss': stop_loss,
            'take_profit': take_profit,
            'entry_sl': entry_sl
        }

        # Check for immediate entry SL exit
        entry_sl_hit = False
        if direction == 'long' and row['low'] <= entry_sl:
            entry_sl_hit = True
            exit_price = entry_sl
        elif direction == 'short' and row['high'] >= entry_sl:
            entry_sl_hit = True
            exit_price = entry_sl

        if entry_sl_hit:
            trade = self.trade_executor.create_trade_record(
                position, exit_price, idx, 'Entry SL', ticker_config
            )
            return None, trade

        return position, None

    def _check_delayed_sl_exit(self, position, row) -> bool:
        """Check if delayed SL exit should trigger"""
        direction = position['direction']
        stop_loss = position['stop_loss']

        if direction == 'long' and row['close'] <= stop_loss:
            return True
        elif direction == 'short' and row['close'] >= stop_loss:
            return True

        return False

    def _handle_exit_on_open(self, position, row, idx, ticker_config) -> Optional[TradeRecord]:
        """Handle exit on next open (delayed SL)"""
        direction = position['direction']
        direction_mult = 1 if direction == 'long' else -1
        exit_price = row['open'] * (1 - self.config.slippage_bps/10000 * direction_mult)

        return self.trade_executor.create_trade_record(
            position, exit_price, idx, 'SL Close Confirmed', ticker_config
        )

    def _manage_position(self, position, row, idx, ticker_config) -> Optional[TradeRecord]:
        """Manage existing position"""
        direction = position['direction']

        # Update trailing stop loss
        if direction == 'long':
            new_sl = row['high'] * (1 - self.config.sl_pct)
            position['stop_loss'] = max(position['stop_loss'], new_sl)
        else:
            new_sl = row['low'] * (1 + self.config.sl_pct)
            position['stop_loss'] = min(position['stop_loss'], new_sl)

        # Update trailing take profit
        if direction == 'long':
            new_tp = row['high'] * (1 - self.config.tp_pct)
            position['take_profit'] = max(position['take_profit'], new_tp)
        else:
            new_tp = row['low'] * (1 + self.config.tp_pct)
            position['take_profit'] = min(position['take_profit'], new_tp)

        # Check exits
        exit_price = None
        exit_reason = None

        # Check take profit
        if direction == 'long' and row['high'] >= position['take_profit']:
            exit_price = position['take_profit']
            exit_reason = 'Trailing TP'
        elif direction == 'short' and row['low'] <= position['take_profit']:
            exit_price = position['take_profit']
            exit_reason = 'Trailing TP'

        # Check stop loss
        elif direction == 'long' and row['low'] <= position['stop_loss']:
            exit_price = position['stop_loss']
            exit_reason = 'Stop Loss'
        elif direction == 'short' and row['high'] >= position['stop_loss']:
            exit_price = position['stop_loss']
            exit_reason = 'Stop Loss'

        if exit_price and exit_reason:
            return self.trade_executor.create_trade_record(
                position, exit_price, idx, exit_reason, ticker_config
            )

        return None

In [ ]:
# ==========================================
# PARALLEL BACKTEST ENGINE CLASS
# ==========================================

class ParallelBacktestEngine:
    """Main engine for running parallel backtests"""

    def __init__(self, config: PortfolioConfig, ticker_configs: List[TickerConfig]):
        self.config = config
        self.ticker_configs = {tc.ticker: tc for tc in ticker_configs}
        self.data_processor = DataProcessor()
        self.logger = logging.getLogger(f"{__name__}.ParallelBacktestEngine")

    def run_backtest(self, data: pd.DataFrame, ticker_col: str = 'Ticker',
                    date_col: str = 'Date') -> pd.DataFrame:
        """Run complete backtest pipeline"""
        self.logger.info("Starting parallel backtest engine")

        # Process input data
        ticker_data = self.data_processor.process_dataframe(data, ticker_col, date_col)
        if not ticker_data:
            self.logger.error("No valid ticker data found")
            return pd.DataFrame()

        # Prepare data for each ticker
        prepared_data = self._prepare_all_ticker_data(ticker_data)
        if not prepared_data:
            self.logger.error("No valid prepared data")
            return pd.DataFrame()

        # Run parallel backtests
        all_trades = self._run_parallel_backtests(prepared_data)

        # Convert to DataFrame
        if all_trades:
            return self._create_results_dataframe(all_trades)
        else:
            self.logger.warning("No trades generated")
            return pd.DataFrame()

    def _prepare_all_ticker_data(self, ticker_data: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
        """Prepare data for all tickers"""
        prepared_data = {}

        for ticker, data in ticker_data.items():
            try:
                if ticker not in self.ticker_configs:
                    self.logger.warning(f"Skipping {ticker}: No configuration found")
                    continue

                ticker_config = self.ticker_configs[ticker]
                enhanced_data = self.data_processor.prepare_ticker_data(data, ticker_config)

                if len(enhanced_data) > 10:
                    prepared_data[ticker] = enhanced_data
                    self.logger.info(f"✅ Prepared {ticker} with {len(enhanced_data)} rows")
                else:
                    self.logger.warning(f"Skipping {ticker}: Insufficient data after processing")

            except Exception as e:
                self.logger.error(f"Error preparing data for {ticker}: {e}")
                continue

        return prepared_data

    def _run_parallel_backtests(self, prepared_data: Dict[str, pd.DataFrame]) -> List[TradeRecord]:
        """Run backtests in parallel"""
        self.logger.info(f"Running parallel backtests on {len(prepared_data)} tickers")

        # Prepare arguments for parallel processing
        backtest_args = []
        for ticker, data in prepared_data.items():
            ticker_config = self.ticker_configs[ticker]
            backtest_args.append((ticker, data, self.config, ticker_config))

        all_trades = []
        max_workers = min(5, len(backtest_args), 8)

        if len(backtest_args) == 1:
            # Single ticker - run directly
            ticker_name, trades = self._backtest_single_ticker(backtest_args[0])
            all_trades.extend(trades)
            self.logger.info(f"Processed {ticker_name}: {len(trades)} trades")
        else:
            # Multiple tickers - use parallel processing
            try:
                with ProcessPoolExecutor(max_workers=max_workers) as executor:
                    future_to_ticker = {
                        executor.submit(backtest_single_ticker_wrapper, args): args[0]
                        for args in backtest_args
                    }

                    for future in as_completed(future_to_ticker):
                        ticker = future_to_ticker[future]
                        try:
                            ticker_name, trades = future.result(timeout=300)
                            all_trades.extend(trades)
                            self.logger.info(f"Processed {ticker_name}: {len(trades)} trades")
                        except Exception as e:
                            self.logger.error(f"Error processing {ticker}: {e}")

            except Exception as e:
                self.logger.error(f"Error in parallel processing: {e}")
                # Fallback to sequential processing
                for args in backtest_args:
                    try:
                        ticker_name, trades = self._backtest_single_ticker(args)
                        all_trades.extend(trades)
                        self.logger.info(f"Processed {ticker_name}: {len(trades)} trades")
                    except Exception as e:
                        self.logger.error(f"Error processing {args[0]}: {e}")

        return all_trades

    def _backtest_single_ticker(self, args: Tuple) -> Tuple[str, List[TradeRecord]]:
        """Run backtest for single ticker"""
        ticker, data, config, ticker_config = args
        strategy = SingleTickerStrategy(config)
        trades = strategy.run_backtest(ticker, data, ticker_config)
        return ticker, trades

    def _create_results_dataframe(self, trades: List[TradeRecord]) -> pd.DataFrame:
        """Convert trade records to DataFrame"""
        try:
            trades_df = pd.DataFrame([
                {
                    'ticker': trade.ticker,
                    'entry_time': trade.entry_time,
                    'exit_time': trade.exit_time,
                    'direction': trade.direction,
                    'entry_price': trade.entry_price,
                    'exit_price': trade.exit_price,
                    'quantity': trade.quantity,
                    'gross_pnl': trade.gross_pnl,
                    'fees': trade.fees,
                    'slippage': trade.slippage,
                    'net_pnl': trade.net_pnl,
                    'stop_loss': trade.stop_loss,
                    'take_profit': trade.take_profit,
                    'exit_reason': trade.exit_reason
                }
                for trade in trades
            ])

            return trades_df.sort_values('entry_time').reset_index(drop=True)

        except Exception as e:
            self.logger.error(f"Error creating results DataFrame: {e}")
            return pd.DataFrame()


In [ ]:
# ==========================================
# RESULTS ANALYZER CLASS
# ==========================================

class ResultsAnalyzer:
    """Analyze and display backtest results"""

    def __init__(self):
        self.logger = logging.getLogger(f"{__name__}.ResultsAnalyzer")

    def analyze_results(self, results: pd.DataFrame) -> Dict:
        """Comprehensive results analysis"""
        if results.empty:
            return {}

        try:
            analysis = {}

            # Overall metrics
            analysis['overall'] = {
                'total_trades': len(results),
                'total_pnl': results['net_pnl'].sum(),
                'win_rate': (results['net_pnl'] > 0).mean(),
                'avg_trade': results['net_pnl'].mean(),
                'best_trade': results['net_pnl'].max(),
                'worst_trade': results['net_pnl'].min(),
                'total_fees': results['fees'].sum(),
                'total_slippage': results['slippage'].sum(),
            }

            # Per-ticker analysis
            analysis['by_ticker'] = {}
            for ticker in results['ticker'].unique():
                ticker_trades = results[results['ticker'] == ticker]
                analysis['by_ticker'][ticker] = {
                    'trades': len(ticker_trades),
                    'pnl': ticker_trades['net_pnl'].sum(),
                    'win_rate': (ticker_trades['net_pnl'] > 0).mean(),
                    'avg_trade': ticker_trades['net_pnl'].mean(),
                    'best_trade': ticker_trades['net_pnl'].max(),
                    'worst_trade': ticker_trades['net_pnl'].min(),
                }

            # Exit reason analysis
            if 'exit_reason' in results.columns:
                analysis['exit_reasons'] = results['exit_reason'].value_counts().to_dict()

            return analysis

        except Exception as e:
            self.logger.error(f"Error analyzing results: {e}")
            return {}

    def print_results_summary(self, results: pd.DataFrame, analysis: Dict):
        """Print comprehensive results summary"""
        print("\n" + "="*60)
        print("ENHANCED BACKTEST RESULTS SUMMARY")
        print("="*60)

        if not analysis or 'overall' not in analysis:
            print("No analysis results available")
            return

        overall = analysis['overall']

        # Overall performance
        print(f"📊 OVERALL PERFORMANCE:")
        print(f"   Total P&L: ${overall.get('total_pnl', 0):,.2f}")
        print(f"   Total Trades: {overall.get('total_trades', 0):,}")
        print(f"   Win Rate: {overall.get('win_rate', 0):.2%}")
        print(f"   Average Trade: ${overall.get('avg_trade', 0):.2f}")
        print(f"   Best Trade: ${overall.get('best_trade', 0):,.2f}")
        print(f"   Worst Trade: ${overall.get('worst_trade', 0):,.2f}")
        print(f"   Total Fees: ${overall.get('total_fees', 0):,.2f}")
        print(f"   Total Slippage: ${overall.get('total_slippage', 0):,.2f}")

        # Per-ticker breakdown
        print(f"\n📈 PER-TICKER BREAKDOWN:")
        print("-" * 50)
        for ticker, metrics in analysis.get('by_ticker', {}).items():
            print(f"   {ticker}:")
            print(f"     Trades: {metrics['trades']:,} | P&L: ${metrics['pnl']:,.2f}")
            print(f"     Win Rate: {metrics['win_rate']:.2%} | Avg: ${metrics['avg_trade']:.2f}")
            print(f"     Best: ${metrics['best_trade']:,.2f} | Worst: ${metrics['worst_trade']:,.2f}")
            print()

        # Exit reasons
        if 'exit_reasons' in analysis:
            print(f"🎯 EXIT REASON BREAKDOWN:")
            print("-" * 30)
            total_trades = sum(analysis['exit_reasons'].values())
            for reason, count in analysis['exit_reasons'].items():
                pct = count / total_trades * 100
                print(f"   {reason}: {count} trades ({pct:.1f}%)")

        # Sample trades
        if not results.empty:
            print(f"\n💼 SAMPLE TRADES (First {min(5, len(results))}):")
            display_cols = ['ticker', 'direction', 'entry_price', 'exit_price', 'net_pnl', 'exit_reason']
            available_cols = [col for col in display_cols if col in results.columns]
            print(results[available_cols].head().to_string(index=False))

        print("="*60)

# ==========================================
# WRAPPER FUNCTIONS FOR MULTIPROCESSING
# ==========================================

def backtest_single_ticker_wrapper(args: Tuple) -> Tuple[str, List[TradeRecord]]:
    """Wrapper function for multiprocessing"""
    ticker, data, config, ticker_config = args
    strategy = SingleTickerStrategy(config)
    trades = strategy.run_backtest(ticker, data, ticker_config)
    return ticker, trades

# ==========================================
# MAIN INTERFACE FUNCTION
# ==========================================

def run_enhanced_backtest(data: pd.DataFrame,
                         ticker_configs: List[TickerConfig],
                         ticker_col: str = 'Ticker',
                         date_col: str = 'Date',
                         config: Optional[PortfolioConfig] = None) -> pd.DataFrame:
    """
    Main function to run enhanced backtest with improved entry/exit logic

    Key Features:
    - Entry SL Check: Immediate exit if price hits entry stop loss on entry bar
    - Delayed SL Exit: Close on next open when stop loss is hit during the bar
    - Trailing Stop Loss: Dynamic updates based on current highs/lows
    - Trailing Take Profit: Dynamic updates based on current highs/lows
    - ATR-based Position Sizing: Using ATR_max calculation
    - Enhanced Risk Management: Daily drawdown limits with proper tracking
    - Parallel Processing: Multi-ticker backtesting with proper logging

    Parameters:
    -----------
    data : pd.DataFrame
        Your DataFrame with OHLCV data
    ticker_configs : List[TickerConfig]
        List of TickerConfig objects with custom parameters for each ticker
    ticker_col : str, default 'Ticker'
        Name of ticker column
    date_col : str, default 'Date'
        Name of date column
    config : PortfolioConfig, optional
        Portfolio configuration (uses default if None)

    Returns:
    --------
    pd.DataFrame
        DataFrame with trade results
    """

    print("🚀 Enhanced Trading Backtest Engine v2.0")
    print("="*60)

    # Set default configuration if not provided
    if config is None:
        config = PortfolioConfig(
            initial_capital=500_000,
            max_risk_per_trade=5_000,
            base_risk_per_trade=1_000,
            daily_drawdown_limit=0.05,
            tp_pct=0.02,
            sl_pct=0.012,
            entry_sl_pct=0.006,
            fee_rate=0.001,
            slippage_bps=2.0
        )

    # Display configuration
    print(f"💰 Portfolio Configuration:")
    print(f"   Initial Capital: ${config.initial_capital:,.2f}")
    print(f"   Max Risk/Trade: ${config.max_risk_per_trade:,.2f}")
    print(f"   Daily Drawdown Limit: {config.daily_drawdown_limit:.1%}")
    print(f"   Take Profit: {config.tp_pct:.1%}")
    print(f"   Stop Loss: {config.sl_pct:.1%}")
    print(f"   Entry Stop Loss: {config.entry_sl_pct:.1%}")
    print("-" * 60)

    print(f"⚙️  Ticker Configurations ({len(ticker_configs)} tickers):")
    for tc in ticker_configs:
        print(f"   📊 {tc.ticker}:")
        print(f"      JMA: Smooth={tc.jma_params.smooth_length}, Norm={tc.jma_params.normalization_period}")
        print(f"      Thresholds: Upper={tc.jma_params.upper_threshold}, Lower={tc.jma_params.lower_threshold}")
        print(f"      Slippage: {tc.slippage_bps} bps")
    print("-" * 60)

    try:
        # Initialize engine and run backtest
        engine = ParallelBacktestEngine(config, ticker_configs)
        results = engine.run_backtest(data, ticker_col, date_col)

        # Analyze and display results
        if not results.empty:
            analyzer = ResultsAnalyzer()
            analysis = analyzer.analyze_results(results)
            analyzer.print_results_summary(results, analysis)

            # Export results
            try:
                results.to_csv('enhanced_backtest_results.csv', index=False)
                print(f"\n💾 Results exported to 'enhanced_backtest_results.csv'")
            except Exception as e:
                print(f"⚠️  Could not export results: {e}")

        else:
            print("\n⚠️  No trades generated. Check:")
            print("   • Signal generation parameters")
            print("   • Data quality and format")
            print("   • Risk management constraints")
            print("   • ATR calculations")

        print(f"\n✅ Enhanced backtest complete!")
        return results

    except Exception as e:
        logger.error(f"Error running enhanced backtest: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def print_usage_guide():
    """Print usage guide for the enhanced backtest system"""
    print("\n" + "="*60)
    print("📖 ENHANCED BACKTEST SYSTEM USAGE GUIDE")
    print("="*60)

    print("\n1️⃣  BASIC USAGE:")
    print("""
    # Create ticker configurations
    ticker_configs = create_example_ticker_configs()

    # Run backtest with your data
    results = run_enhanced_backtest(
        data=your_dataframe,
        ticker_configs=ticker_configs,
        ticker_col='Ticker',
        date_col='Date'
    )
    """)

    print("\n2️⃣  CUSTOM CONFIGURATION:")
    print("""
    # Create custom portfolio config
    custom_config = PortfolioConfig(
        initial_capital=1_000_000,
        max_risk_per_trade=10_000,
        daily_drawdown_limit=0.02,
        tp_pct=0.03,
        sl_pct=0.015
    )

    # Run with custom config
    results = run_enhanced_backtest(
        data=your_dataframe,
        ticker_configs=ticker_configs,
        config=custom_config
    )
    """)

    print("\n3️⃣  DATA REQUIREMENTS:")
    print("""
    Your DataFrame should have:
    • Ticker column (default: 'Ticker')
    • Date column (default: 'Date')
    • OHLC columns: 'open', 'high', 'low', 'close'
    • Volume column (optional)
    """)

    print("\n4️⃣  KEY FEATURES:")
    print("""
    ✅ ATR-based position sizing
    ✅ Entry stop loss checks
    ✅ Delayed stop loss exits
    ✅ Trailing stops and take profits
    ✅ Daily drawdown management
    ✅ Parallel multi-ticker processing
    ✅ Comprehensive logging
    ✅ Detailed performance analysis
    """)

    print("="*60)

# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    print_usage_guide()

    print("\n" + "="*60)
    print("🎯 EXAMPLE DEMO")
    print("="*60)

    # Create example configurations
    example_configs = create_example_configs()

    # Create enhanced portfolio configuration
    demo_config = PortfolioConfig(
        initial_capital=500_000,
        max_risk_per_trade=5_000,
        base_risk_per_trade=1_000,
        daily_drawdown_limit=0.02,  # 2%
        tp_pct=0.025,  # 2.5%
        sl_pct=0.015,  # 1.5%
        entry_sl_pct=0.008,  # 0.8%
        fee_rate=0.001,
        slippage_bps=2.0
    )

    print("💡 To use with your actual data:")
    print("""
    # Load your data
    your_data = pd.read_csv('your_data.csv')

    # Create ticker configs
    ticker_configs = create_example_ticker_configs()

    # Run backtest
    results = run_enhanced_backtest(
        data=your_data,
        ticker_configs=ticker_configs,
        config=demo_config
    )

    # Analyze results
    if not results.empty:
        analyzer = ResultsAnalyzer()
        analysis = analyzer.analyze_results(results)
        analyzer.print_results_summary(results, analysis)
    """)

    print("\n🎉 Enhanced Trading Backtest System Ready!")
    print("="*60)